# DE3 — Time-based train/test split with leakage checks

**Project:** DS 5020 Mini-Project "Customers Like You" (PCA/SVD recommender on UCI Online Retail II).

Input: `data/processed/clean.parquet` (output of DE2). Logic lives in `src/split.py`; this notebook calls it,
prints the report, runs the leakage assertions and writes the artefacts.

Rules:

1. `cutoff = (max InvoiceDate - 2 months).normalize()`; **train** = rows before the cutoff, **test period** = rows on/after it.
2. **Targets** = `(Customer ID, StockCode)` pairs bought in the test period that the customer **never** bought in train.
   Repeat purchases are dropped from the targets.
3. **Evaluation customers** = customers present in train *and* with at least one target. Test customers with no training
   history are **cold-start** and are excluded (count reported).
4. **No popularity filter here** (that is DE4, fitted on train only). Each target is flagged `seen_in_train`
   (bought by *anyone* in train) so we know which targets are winnable at all.

Outputs (both gitignored): `data/processed/train.parquet`, `data/processed/test_targets.parquet`.

In [1]:
# Environment guard: this notebook must run in the "giftlane" conda env (pandas 3.0.6).
# It fails fast with "Wrong Python environment. Run: conda activate giftlane" otherwise.
import sys
from pathlib import Path

_ROOT = Path.cwd().resolve()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from src.env_check import check_env

print("environment OK:", check_env())

environment OK: /opt/homebrew/Caskroom/miniforge/base/envs/giftlane/bin/python


## 0. Setup

Resolve the repo root, import `src.split`, load the cleaned data.

In [2]:
import subprocess
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.split import TEST_MONTHS, leakage_checks, print_report, save_outputs, split, top_train_products

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

clean_df = pd.read_parquet(ROOT / "data" / "processed" / "clean.parquet")
print("python :", sys.executable)
print("pandas :", pd.__version__)
print(f"clean rows: {len(clean_df):,} | customers: {clean_df['Customer ID'].nunique():,} | "
      f"products: {clean_df['StockCode'].nunique():,}")
print(f"date range: {clean_df['InvoiceDate'].min()} -> {clean_df['InvoiceDate'].max()}")
print(clean_df.dtypes.to_string())

python : /opt/homebrew/Caskroom/miniforge/base/envs/giftlane/bin/python
pandas : 3.0.6
clean rows: 776,596 | customers: 5,852 | products: 4,621
date range: 2009-12-01 07:45:00 -> 2011-12-09 12:50:00
Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             int64
Country                   str


## 1. Cutoff and split

The cutoff is the last `2` months of the data per the brief, snapped to midnight so the boundary is a calendar date
rather than an arbitrary minute.

In [3]:
result = split(clean_df, months=TEST_MONTHS)
print("cutoff:", result.cutoff)
print(f"train rows      : {len(result.train):,}")
print(f"test period rows: {len(result.test_period):,}")
assert len(result.train) + len(result.test_period) == len(clean_df)

cutoff: 2011-10-09 00:00:00
train rows      : 659,830
test period rows: 116,766


## 2. Leakage assertions

All of these must pass before any model is fitted:

- `train.InvoiceDate.max() < cutoff <= test_period.InvoiceDate.min()`
- no `(customer, StockCode)` target pair exists in train
- every evaluation customer exists in train

(`leakage_checks` raises `AssertionError` on the first failure.)

In [4]:
for msg in leakage_checks(result):
    print("PASS:", msg)

PASS: train max 2011-10-07 16:06:00 < cutoff 2011-10-09 00:00:00 <= test min 2011-10-09 10:13:00
PASS: no (customer, product) target pair exists in train
PASS: every evaluation customer exists in train
PASS: targets are unique pairs and belong only to evaluation customers
PASS: cold-start customers are absent from train


## 3. Split report

Sizes and date ranges of both halves, cold-start exclusions, target statistics, and the share of targets whose product
was never sold in train. Those last targets are unwinnable for *any* model (and for the popularity baseline) and
cap the achievable recall.

In [5]:
print_report(result)

cutoff                      : 2011-10-09 00:00:00  (max InvoiceDate - 2 months, normalised)

TRAIN rows                  : 659,830
      date range            : 2009-12-01 07:45:00  ->  2011-10-07 16:06:00
      customers             : 5,472
      products              : 4,531

TEST PERIOD rows            : 116,766
      date range            : 2011-10-09 10:13:00  ->  2011-12-09 12:50:00
      customers             : 2,423
      distinct (cust, prod) : 95,278  of which repeat purchases: 31,791

cold-start customers excluded (in test, not in train) : 380  (13,840 new pairs dropped with them)
repeat-only customers excluded (in train, no new pair): 112

evaluation customers        : 1,931
total target pairs          : 49,647
targets per eval customer   : mean 25.71, median 17

target pairs with product never seen in train : 3,670 (7.39%)  <- unwinnable for any model trained on train
evaluation customers whose targets are ALL unseen : 14


### Targets per evaluation customer

Distribution of the number of new products each evaluation customer bought in the test period.

In [6]:
per_cust = result.targets.groupby("Customer ID").size()
print(per_cust.describe().round(2).to_string())
print()
print("quantiles:", per_cust.quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).to_dict())
display(result.targets.head(10).drop(columns=["Customer ID"]))  # IDs hidden (R4)

count    1931.00
mean       25.71
std        28.97
min         1.00
25%         8.00
50%        17.00
75%        33.00
max       462.00

quantiles: {0.1: 3.0, 0.25: 8.0, 0.5: 17.0, 0.75: 33.0, 0.9: 58.0, 0.99: 124.50000000000023}


,StockCode,seen_in_train
0,21265,True
1,21636,True
2,22131,True
3,22252,True
4,22561,True
5,22621,True
6,22945,True
7,23271,True
8,23420,True
9,23421,True


## 4. Baseline input: top-15 best sellers in TRAIN by distinct customers

Computed from **train only** — never from the test period. This is the input the popularity baseline will
recommend to everyone in DE4/DE5.

In [7]:
top15 = top_train_products(result.train, n=15)
display(top15)

,StockCode,Description,customers
rank,,,
1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,1422
2,22423,REGENCY CAKESTAND 3 TIER,1213
3,21212,PACK OF 72 RETRO SPOT CAKE CASES,1058
4,22138,BAKING SET 9 PIECE RETROSPOT,975
5,84879,ASSORTED COLOUR BIRD ORNAMENT,907
6,85099B,JUMBO BAG RED RETROSPOT,898
7,47566,PARTY BUNTING,855
8,22139,RETROSPOT TEA SET CERAMIC 11 PC,836
9,20725,LUNCH BAG RED RETROSPOT,805


## 5. Save artefacts and confirm they are gitignored

`train.parquet` holds all train rows; `test_targets.parquet` holds `Customer ID`, `StockCode`, `seen_in_train`.

In [8]:
train_path, targets_path = save_outputs(result, ROOT)
for p in (train_path, targets_path):
    print(f"saved -> {p.relative_to(ROOT)} ({p.stat().st_size / 1e6:.2f} MB)")

chk_train = pd.read_parquet(train_path)
chk_targets = pd.read_parquet(targets_path)
assert len(chk_train) == len(result.train)
assert list(chk_targets.columns) == ["Customer ID", "StockCode", "seen_in_train"]
assert len(chk_targets) == len(result.targets)
print(f"round-trip OK: train {len(chk_train):,} rows; targets {len(chk_targets):,} rows, columns {list(chk_targets.columns)}")
print(chk_targets.dtypes.to_string())

try:
    out = subprocess.run(
        ["git", "check-ignore", "-v", str(train_path.relative_to(ROOT)), str(targets_path.relative_to(ROOT))],
        cwd=ROOT, capture_output=True, text=True, check=True,
    )
    print("\ngit check-ignore:\n" + out.stdout.strip())
except Exception as e:  # git not available / not a repo
    print("git check-ignore could not be run:", e)

saved -> data/processed/train.parquet (4.30 MB)
saved -> data/processed/test_targets.parquet (0.11 MB)
round-trip OK: train 659,830 rows; targets 49,647 rows, columns ['Customer ID', 'StockCode', 'seen_in_train']
Customer ID      int64
StockCode          str
seen_in_train     bool

git check-ignore:
.gitignore:3:data/*	data/processed/train.parquet
.gitignore:3:data/*	data/processed/test_targets.parquet


## Notes / risks

- The test period (2011-10-09 → 2011-12-09) is the Oct–Dec Christmas peak; seasonal demand may favour or hurt both
  the model and the popularity baseline. Flag in the memo.
- Repeat purchases are excluded from targets so that the metric rewards *discovery* of new products, which is what a
  "customers like you" recommender is for; predicting re-buys is a different (easier) task.
- Cold-start customers have no row in the customer × product matrix, so no collaborative model can score them;
  they are reported but excluded from evaluation.